# 01 — Exploratory Data Analysis

**Dataset**: Sparkov Credit Card Fraud — ~1.3M training transactions  
**Target**: `is_fraud` (binary: 0 = legitimate, 1 = fraud)  
**Key Challenge**: Severe class imbalance — ~0.57% fraud

---

### Table of Contents
1. [Setup & Load](#setup)
2. [Dataset Overview & Schema](#schema)
3. [Class Imbalance Analysis](#imbalance) ← **Critical**
4. [Transaction Amount Distribution](#amount)
5. [Temporal Patterns](#temporal)
6. [Geographic Analysis](#geo)
7. [Merchant Category Analysis](#category)
8. [Demographics: Age & Gender](#demographics)
9. [Correlation with Fraud](#correlation)
10. [Key Findings](#findings)

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.figsize': (14, 5), 'figure.dpi': 110,
    'axes.spines.top': False, 'axes.spines.right': False,
    'font.size': 11,
})
FRAUD_PAL = {0: '#2196F3', 1: '#F44336'}  # Blue=Legit, Red=Fraud

from src.data.loader import load_train_test
from src.data.features import engineer_features

df_train, df_test = load_train_test(
    '../data/fraudTrain.csv',
    '../data/fraudTest.csv',
    train_sample_size=200_000,  # Stratified subsample for EDA
)
print(f'Train: {df_train.shape} | Test: {df_test.shape}')

## 2. Dataset Schema & Quality <a id='schema'></a>

In [ ]:
print('=== DTYPES ==='); print(df_train.dtypes)
print('\n=== MISSING VALUES ===')
missing = df_train.isnull().sum()
print(missing[missing > 0] if missing.any() else 'No missing values ✅')
print(f'\nDuplicates: {df_train.duplicated().sum():,}')
df_train.describe().round(2)

## 3. Class Imbalance Analysis ⚠️ <a id='imbalance'></a>

This is the **most important characteristic** of fraud datasets — it fundamentally affects:
- Which metrics to use (PR-AUC > ROC-AUC, F2 > Accuracy)
- Which algorithms to choose (class_weight, scale_pos_weight)
- Threshold selection strategy

In [ ]:
train_fraud_rate = df_train['is_fraud'].mean()
test_fraud_rate  = df_test['is_fraud'].mean()
train_counts = df_train['is_fraud'].value_counts()

print('=== IMBALANCE SUMMARY ===')
print(f'Train fraud rate : {train_fraud_rate:.4%}')
print(f'Test fraud rate  : {test_fraud_rate:.4%}')
print(f'Imbalance ratio  : ~1:{int(1/train_fraud_rate)} (fraud:legit)')
print(f'\nTrain class counts:')
print(f'  Legitimate: {train_counts[0]:>10,}')
print(f'  Fraud:      {train_counts[1]:>10,} ← {train_fraud_rate:.2%} of total')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Log-scale bar chart
axes[0].bar(['Legit (0)', 'Fraud (1)'], train_counts.values,
            color=['#2196F3', '#F44336'], alpha=0.85, edgecolor='white')
axes[0].set_yscale('log')
axes[0].set_ylabel('Count (log scale)')
axes[0].set_title('Class Distribution\n(Log scale)', fontweight='bold')
for bar, (v, label) in zip(axes[0].patches, zip(train_counts.values, ['Legitimate', 'Fraud'])):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()*1.5,
                 f'{v:,}', ha='center', fontsize=11, fontweight='bold')

# Pie chart (emphasizes the imbalance visually)
axes[1].pie(
    [train_counts[0], train_counts[1]],
    labels=[f'Legit ({train_counts[0]/len(df_train):.2%})', f'Fraud ({train_fraud_rate:.2%})'],
    colors=['#2196F3', '#F44336'], autopct=None, startangle=90,
    wedgeprops={'alpha': 0.85, 'edgecolor': 'white', 'linewidth': 2},
)
axes[1].set_title('Class Proportion\n(Almost invisible fraud slice!)', fontweight='bold')

# Train vs test fraud rate comparison
axes[2].bar(['Train', 'Test'], [train_fraud_rate*100, test_fraud_rate*100],
            color=['#4CAF50', '#FF9800'], alpha=0.85, width=0.4)
axes[2].set_ylabel('Fraud Rate (%)')
axes[2].set_title('Fraud Rate: Train vs Test\n(Should be similar)', fontweight='bold')
for i, v in enumerate([train_fraud_rate*100, test_fraud_rate*100]):
    axes[2].text(i, v + 0.02, f'{v:.3f}%', ha='center', fontsize=11, fontweight='bold')

plt.suptitle('⚠️  Class Imbalance Analysis — Fraud Detection Challenge',
             fontsize=13, fontweight='bold', color='#F44336')
plt.tight_layout(); plt.show()

In [ ]:
# Why accuracy is USELESS for fraud detection
print('📊 Why NOT to use Accuracy:')
print(f'   Predicting ALL transactions as "Legitimate" gives {1-train_fraud_rate:.2%} accuracy')
print(f'   ... but catches ZERO fraud!')
print()
print('✅ Correct metrics for fraud detection:')
print('   1. PR-AUC   — Precision-Recall Area Under Curve (primary)')
print('   2. F2-Score — Weights Recall 2× (missing fraud = worse than false alarm)')
print('   3. Recall   — % of actual fraud caught (business critical)')
print('   4. Precision — % of fraud alerts that are real fraud (operational cost)')

## 4. Transaction Amount Analysis <a id='amount'></a>

In [ ]:
fraud_amt  = df_train[df_train['is_fraud']==1]['amt']
legit_amt  = df_train[df_train['is_fraud']==0]['amt']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Raw distribution (log scale x-axis)
axes[0].hist(legit_amt.clip(0, 2000), bins=80, color='#2196F3', alpha=0.6, density=True, label='Legit')
axes[0].hist(fraud_amt.clip(0, 2000), bins=80, color='#F44336', alpha=0.6, density=True, label='Fraud')
axes[0].set_xlabel('Transaction Amount ($)')
axes[0].set_title('Amount Distribution (clipped at $2K)', fontweight='bold')
axes[0].legend()

# Boxplot
axes[1].boxplot([legit_amt.clip(0, 1000), fraud_amt.clip(0, 1000)],
                labels=['Legitimate', 'Fraud'],
                patch_artist=True, showfliers=False,
                boxprops={'facecolor': '#E3F2FD'},
                medianprops={'color': '#F44336', 'lw': 2})
axes[1].set_ylabel('Transaction Amount ($)')
axes[1].set_title('Amount Boxplot (IQR, no outliers)', fontweight='bold')

# Mean/Median comparison
stats = pd.DataFrame({
    'Metric': ['Mean', 'Median', 'Std', '95th pct'],
    'Legitimate': [
        legit_amt.mean(), legit_amt.median(), legit_amt.std(), legit_amt.quantile(0.95)
    ],
    'Fraud': [
        fraud_amt.mean(), fraud_amt.median(), fraud_amt.std(), fraud_amt.quantile(0.95)
    ]
}).set_index('Metric').round(2)
print('Transaction Amount Statistics:')
print(stats.to_string())

axes[2].bar(['Legit Mean', 'Fraud Mean'], [legit_amt.mean(), fraud_amt.mean()],
            color=['#2196F3', '#F44336'], alpha=0.85, width=0.4)
axes[2].set_ylabel('Mean Amount ($)')
axes[2].set_title('Mean Transaction Amount', fontweight='bold')
for i, v in enumerate([legit_amt.mean(), fraud_amt.mean()]):
    axes[2].text(i, v + 2, f'${v:.2f}', ha='center', fontweight='bold')

plt.suptitle('Transaction Amount Analysis by Fraud Status', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

## 5. Temporal Patterns <a id='temporal'></a>

In [ ]:
ts = pd.to_datetime(df_train['trans_date_trans_time'], errors='coerce')
df_train['hour']       = ts.dt.hour
df_train['day_of_week']= ts.dt.dayofweek
df_train['month']      = ts.dt.month

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, col, title, xlabels in zip(
    axes,
    ['hour', 'day_of_week', 'month'],
    ['Fraud Rate by Hour of Day', 'Fraud Rate by Day of Week', 'Fraud Rate by Month'],
    [None, ['Mon','Tue','Wed','Thu','Fri','Sat','Sun'], None]
):
    fraud_rate = df_train.groupby(col)['is_fraud'].mean() * 100
    bars = ax.bar(fraud_rate.index, fraud_rate.values, color='#F44336', alpha=0.75)
    ax.axhline(train_fraud_rate * 100, color='grey', ls='--', lw=1.5, label='Average')
    ax.set_xlabel(col.replace('_', ' ').title())
    ax.set_ylabel('Fraud Rate (%)')
    ax.set_title(title, fontweight='bold')
    ax.legend(fontsize=9)
    if xlabels:
        ax.set_xticks(range(len(xlabels)))
        ax.set_xticklabels(xlabels, rotation=20)

plt.suptitle('Temporal Patterns — When Does Fraud Happen?', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

## 6. Geographic Distance Analysis <a id='geo'></a>

In [ ]:
from src.data.features import haversine_km

lat = pd.to_numeric(df_train['lat'], errors='coerce').fillna(0)
lon = pd.to_numeric(df_train['long'], errors='coerce').fillna(0)
merch_lat = pd.to_numeric(df_train['merch_lat'], errors='coerce').fillna(0)
merch_lon = pd.to_numeric(df_train['merch_long'], errors='coerce').fillna(0)

dist_km = haversine_km(lat.values, lon.values, merch_lat.values, merch_lon.values)
df_train['distance_km'] = dist_km

fraud_dist  = df_train[df_train['is_fraud']==1]['distance_km']
legit_dist  = df_train[df_train['is_fraud']==0]['distance_km']

print(f'Legitimate — median distance: {legit_dist.median():.1f} km | mean: {legit_dist.mean():.1f} km')
print(f'Fraud       — median distance: {fraud_dist.median():.1f} km | mean: {fraud_dist.mean():.1f} km')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(np.log1p(legit_dist.clip(0, 500)), bins=60, alpha=0.6, color='#2196F3', density=True, label='Legit')
axes[0].hist(np.log1p(fraud_dist.clip(0, 500)), bins=60, alpha=0.6, color='#F44336', density=True, label='Fraud')
axes[0].set_xlabel('log1p(Distance km)')
axes[0].set_title('Customer-Merchant Distance Distribution', fontweight='bold')
axes[0].legend()

# Distance decile vs fraud rate
df_train['dist_decile'] = pd.qcut(df_train['distance_km'], q=10, labels=False, duplicates='drop')
dist_fraud_rate = df_train.groupby('dist_decile')['is_fraud'].mean() * 100
axes[1].bar(range(len(dist_fraud_rate)), dist_fraud_rate.values, color='#F44336', alpha=0.75)
axes[1].set_xlabel('Distance Decile (0=closest, 9=farthest)')
axes[1].set_ylabel('Fraud Rate (%)')
axes[1].set_title('Fraud Rate by Distance Decile', fontweight='bold')

plt.suptitle('Geographic Distance: Customer ↔ Merchant', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

## 7. Merchant Category Analysis <a id='category'></a>

In [ ]:
cat_stats = df_train.groupby('category').agg(
    n_transactions=('is_fraud', 'count'),
    fraud_rate=('is_fraud', 'mean'),
    avg_amount=('amt', 'mean'),
).reset_index().sort_values('fraud_rate', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Fraud rate by category
bars = axes[0].barh(cat_stats['category'][::-1], cat_stats['fraud_rate'][::-1]*100,
                     color='#F44336', alpha=0.8)
axes[0].axvline(train_fraud_rate*100, color='grey', ls='--', lw=1.5, label='Overall avg')
axes[0].set_xlabel('Fraud Rate (%)')
axes[0].set_title('Fraud Rate by Merchant Category', fontweight='bold')
for bar, v in zip(bars[::-1], cat_stats['fraud_rate']):
    axes[0].text(v*100 + 0.02, bar.get_y()+bar.get_height()/2,
                 f'{v:.2%}', va='center', fontsize=9)
axes[0].legend()

# Transaction volume by category (bubble = fraud count)
axes[1].scatter(
    cat_stats['avg_amount'],
    cat_stats['fraud_rate']*100,
    s=cat_stats['n_transactions']/50,
    c=cat_stats['fraud_rate'], cmap='RdYlGn_r', alpha=0.8, edgecolors='grey', lw=0.5
)
for _, row in cat_stats.iterrows():
    axes[1].annotate(row['category'].replace('_', '\n'), (row['avg_amount'], row['fraud_rate']*100),
                     fontsize=7, ha='center')
axes[1].set_xlabel('Average Transaction Amount ($)')
axes[1].set_ylabel('Fraud Rate (%)')
axes[1].set_title('Fraud Rate vs Avg Amount\n(bubble size = transaction volume)', fontweight='bold')

plt.suptitle('Merchant Category Analysis', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()
print(cat_stats.to_string(index=False))

## 8. Demographics: Age & Gender <a id='demographics'></a>

In [ ]:
ts = pd.to_datetime(df_train['trans_date_trans_time'], errors='coerce')
dob = pd.to_datetime(df_train['dob'], errors='coerce')
df_train['age'] = ((ts - dob).dt.days / 365.25).clip(0, 100)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Age distribution
for label, color in FRAUD_PAL.items():
    axes[0].hist(df_train[df_train['is_fraud']==label]['age'].dropna(),
                 bins=40, alpha=0.6, color=color, density=True,
                 label=['Legit', 'Fraud'][label])
axes[0].set_xlabel('Customer Age (years)')
axes[0].set_title('Age Distribution by Fraud Status', fontweight='bold')
axes[0].legend()

# Age decile fraud rate
df_train['age_group'] = pd.cut(df_train['age'], bins=[0,25,35,45,55,65,100],
                                labels=['<25','25-35','35-45','45-55','55-65','65+'])
age_fraud = df_train.groupby('age_group', observed=True)['is_fraud'].mean() * 100
axes[1].bar(age_fraud.index, age_fraud.values, color='#FF9800', alpha=0.85)
axes[1].set_xlabel('Age Group')
axes[1].set_ylabel('Fraud Rate (%)')
axes[1].set_title('Fraud Rate by Age Group', fontweight='bold')

# Gender fraud rate
gender_stats = df_train.groupby('gender')['is_fraud'].agg(['mean', 'count'])
gender_stats['fraud_rate_pct'] = gender_stats['mean'] * 100
axes[2].bar(gender_stats.index, gender_stats['fraud_rate_pct'],
            color=['#E91E63', '#2196F3'], alpha=0.85, width=0.4)
axes[2].set_ylabel('Fraud Rate (%)')
axes[2].set_title('Fraud Rate by Gender', fontweight='bold')
for i, (g, row) in enumerate(gender_stats.iterrows()):
    axes[2].text(i, row['fraud_rate_pct']+0.02,
                 f"{row['fraud_rate_pct']:.3f}%\n(n={row['count']:,})",
                 ha='center', fontsize=10)

plt.suptitle('Demographic Analysis', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

## 10. Key Findings & Hypotheses <a id='findings'></a>

| # | Finding | Strength | Implication |
|---|---------|----------|-------------|
| 1 | **Severe class imbalance** (~0.57% fraud) | ⭐⭐⭐ | Use PR-AUC & F2, not accuracy; need sampling/class_weight |
| 2 | **Geographic distance**: Fraud has larger customer-merchant distance | ⭐⭐⭐ | Haversine distance is a strong feature |
| 3 | **Transaction amount**: Fraud tends to be higher (but overlapping) | ⭐⭐ | log_amt + amt_to_pop_ratio are key features |
| 4 | **Time of night** (11pm-5am): Elevated fraud rate | ⭐⭐ | is_night binary feature |
| 5 | **Merchant category** varies significantly in fraud rate | ⭐⭐ | Category is a strong categorical predictor |
| 6 | **Gender** has minimal difference in fraud rate | ⭐ | Low predictive value |

> **Next**: → `02_Feature_Engineering.ipynb`